In [0]:
%pip install beautifulsoup4
try:
 from bs4 import BeautifulSoup
 print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:
 print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")

In [0]:
# CELL 2: Setup & Config
import os
from datetime import datetime, timedelta

# Azure Storage Config
storage_account_name = "REMOVED_FOR_SECURITY"
storage_account_key = "TSub+REMOVED_FOR_SECURITY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

# Define Folders
landing_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/"
archive_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/archive/"

print("✅ Configuration Loaded.")

In [0]:
# def cleanup_old_files(days_to_keep=14):
#     print("🧹 Checking archive for old files...")
#     cutoff_date = datetime.now() - timedelta(days=days_to_keep)
    
#     try:
#         files = dbutils.fs.ls(archive_zone)
#         for f in files:
#             # Expected format: YYYY-MM-DD.csv
#             filename = f.name.replace(".csv", "")
#             try:
#                 file_date = datetime.strptime(filename, "%Y-%m-%d")
#                 if file_date < cutoff_date:
#                     print(f"🗑️ Deleting: {f.name}")
#                     dbutils.fs.rm(f.path)
#             except ValueError:
#                 continue
#     except Exception:
#         print("Archive is empty or does not exist yet.")

# cleanup_old_files()

# CELL 3: Retention Policy (Skipped - using manual snapshot)
print("⏭️ Skipping retention policy - using manual data snapshot.")

In [0]:
# def archive_current_file():
#     current_file_path = f"{landing_zone}current.csv"
    
#     try:
#         # Check if file exists
#         dbutils.fs.ls(current_file_path)
        
#         # Rename to YYYY-MM-DD.csv
#         today_str = datetime.now().strftime("%Y-%m-%d")
#         archive_path = f"{archive_zone}{today_str}.csv"
        
#         print(f"📦 Archiving previous 'current.csv' to: {archive_path}")
#         dbutils.fs.mv(current_file_path, archive_path)
        
#     except Exception:
#         print("No 'current.csv' found in landing. First run?")

# archive_current_file()
# CELL 4: Archival Logic (Skipped - using manual snapshot)
print("⏭️ Skipping archival - using manual data snapshot.")

In [0]:
# import os
# import time

# def download_snapshot_with_retry(max_retries=3):
#     fresh_url = get_dynamic_download_url()
#     local_path = "/tmp/current.csv"
#     headers = {'User-Agent': 'Mozilla/5.0'}

#     for attempt in range(1, max_retries + 1):
#         try:
#             print(f"⬇️ Attempt {attempt}/{max_retries}: Downloading to driver node...")

#             # 30-second timeout, Stream=True
#             with requests.get(fresh_url, headers=headers, stream=True, timeout=30) as r:
#                 r.raise_for_status()
#                 with open(local_path, 'wb') as f:
#                     # 10MB Chunks
#                     for chunk in r.iter_content(chunk_size=10*1024*1024):
#                         if chunk:
#                             f.write(chunk)

#             print("✅ Download complete.")
#             break  # Exit loop on success

#         except Exception as e:
#             print(f"⚠️ Attempt {attempt} failed: {str(e)}")
#             if attempt < max_retries:
#                 print("⏳ Waiting 5 seconds...")
#                 time.sleep(5)
#             else:
#                 print("❌ All retries failed.")
#                 raise e

#     # Upload to Data Lake
#     if os.path.exists(local_path):
#         print(f"⬆️ Uploading to Bronze Landing Zone...")
#         dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
#         os.remove(local_path)
#         print("✅ Success! 'current.csv' is updated in the Data Lake.")

# # Run it
# download_snapshot_with_retry()

# CELL 5: Verify files exist in Bronze layer
print("🔍 Verifying manual snapshot files in Bronze layer...")

files_to_check = [
    f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/current.csv",
    f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/units/landing/current.csv",
    f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/projects/landing/current.csv"
]

all_good = True
for file_path in files_to_check:
    try:
        dbutils.fs.ls(file_path)
        print(f"✅ Found: {file_path.split('/')[-3]}/landing/current.csv")
    except Exception:
        print(f"❌ MISSING: {file_path}")
        all_good = False

if all_good:
    print("\n🚀 All files verified! Pipeline can proceed.")
else:
    print("\n⚠️ Some files are missing. Please upload them manually.")